# Identifying topics using regex search and zero-shot modelling

This workbook follows on from `1_0_2_topic_modelling.ipynb', to explore other methods of identifying the topics/themes of comments.

Here I assume I already know the topic/themes which I'm interested in identifying. Given a list of these themes I then try to identify them in the text, using either:
* Regex matching (super low tech, just matching based on stem words)
* Fuzzy matching (still super low tech, but allowing for spelling mistakes)
* Comparing the cosine similarity of embeddings 
* Zero shot classification using the HuggingFace transformer to provide the embeddings. 

In [ ]:
import pandas as pd
import numpy as np
import re
from collections import defaultdict
import string 
from datasets import Dataset
import regex

from transformers import AutoModelForMaskedLM, pipeline, AutoTokenizer, AutoModel 
import torch
from sentence_transformers import SentenceTransformer

from torch.nn.functional import cosine_similarity

import sys
sys.path.append('../pipeline')
from nlp_tasks import NLP_Tasks
from comments_saver import CommentsSaver

In [11]:
# Load data
df = pd.read_csv('../outputs/train_comments.csv')

### Some pre-processing of the text 
Some very minimal pre-processing of the text - trying not to change the structure just tidy it a little bit. 

In [13]:
nlp = NLP_Tasks(model_checkpoint)

# remove place names, numbers and non-ASCII characters
# df = nlp.remove_place_names(df=df, column='comment_text')
# df = nlp.remove_numbers(df=df, column='cleaned_comment_text')
df = nlp.remove_non_ascii(df=df, column='comment_text')

Device set to use mps:0


In [14]:
# remove punctuation except for periods
def remove_punctuation_except_period(text):
    # Keep periods, remove all other punctuation
    punctuation_to_remove = string.punctuation.replace('.', '')
    translator = str.maketrans('', '', punctuation_to_remove)
    return text.translate(translator)

df['cleaned_comment_text'] = df['comment_text'].apply(remove_punctuation_except_period)

In [15]:
# remove '\n' and remove extra spaces
def remove_newlines_and_extra_spaces(text):
    # Remove newlines and extra spaces
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df['cleaned_comment_text'] = df['cleaned_comment_text'].apply(remove_newlines_and_extra_spaces)

In [16]:
# # remove the phrase 'Not Available'
# def remove_not_available(text):
#     # Remove the phrase 'Not Available'
#     text = text.replace('Not Available', '')
#     # Remove any leading or trailing whitespace
#     text = text.strip()
#     return text

# df['cleaned_comment_text'] = df['cleaned_comment_text'].apply(remove_not_available)

In [17]:
# remove rows with empty strings
df = df[df['cleaned_comment_text'].str.strip() != '']
# remove rows with empty strings
df = df[df['cleaned_comment_text'].str.strip() != 'Not Available']
# remove rows with empty strings
df = df[df['cleaned_comment_text'].str.strip() != 'not available']

In [18]:
# cleaned_place_text = df_split['cleaned_comment_text'].tolist()
cleaned_place_text = df['cleaned_comment_text'].tolist()

In [19]:
# remove duplicates from cleaned_place_text
cleaned_place_text = list(set(cleaned_place_text))

In [20]:
print(f'Total number of unique Ealing comments: {len(cleaned_place_text)}')

Total number of unique Ealing comments: 514


### Define the topics

Define both topic labels and lists of strings for each label. Given the list of topiuc labels I generated the lists using ChatGPT. 

I've added in one 'rogue' topic label of "pizza", as if this is flagged as relevant then there's definietly something wrong. 

In [ ]:
topic_labels = [
    "character & setting",
    "scale",
    "density",
    "heritage",
    "housing mix",
    "affordability",
    "amenity impact",
    "infrastructure",
    "traffic / highways",
    "air pollution",
    "green space loss",
    "parking",
    "highway safety",
    "consultation",
    "accessibility", 
    "drainage / flooding",
    "pizza" # one completely irrelevant topic to see if it works 
]

topic_list = [
    ["character", "appearance", "visual impact", "design", "setting", "fit with surroundings", "look of local area"],
    ["scale", "size of development", "large-scale", "massive", "overdevelopment"],
    ["density", "overcrowding", "high-rise", "compact", "too many units", "number of dwellings"],
    ["heritage", "designated heritage", "nondesignated heritage", "historical setting", "listed building", "conservation area"],
    ["housing mix", "family housing", "dwelling types", "type of homes", "house types", "incompatible housing"],
    ["affordable", "not affordable", "housing cost", "unaffordable", "affordability"],
    ["amenity", "neighbouring properties", "neighbor impact", "residential amenity", "privacy", "light", "noise"],
    ["infrastructure", "schools", "utilities", "parks", "public services", "healthcare", "surgeries"],
    ["traffic", "highway network", "transport", "road congestion", "rush hour", "traffic jam"],
    ["air pollution", "pollution", "emissions", "bad air", "polluted"],
    ["trees", "green space", "loss of trees", "vegetation", "natural landscape", "greenery"],
    ["parking", "onsite car parking", "car space", "loss of parking", "parking pressure"],
    ["highway safety", "footpath", "pedestrian", "road safety", "estate road"],
    ["public consultation", "community input", "lack of consultation", "inadequate engagement"],
    ["disability", "wheelchair", "not accessible", "disabled access", "mobility issues", "not suitable for disabilities"],
    ["drainage", "flooding", "surface water", "sewage", "runoff", "water management", "neighboring drainage"],
    ["pizza", "pepperoni pizza", "pizza toppings", "pizza crust", "pizza delivery", "pizza restaurant", "pizza place"]
]

In [31]:
# Compile patterns for each topic
compiled_patterns = []
for keywords in topic_list:
    pattern = r"\b(" + "|".join(re.escape(word) for word in keywords) + r")\b"
    compiled_patterns.append(re.compile(pattern, re.IGNORECASE))

### Basic regex match 

Return a count of the regex matches. 

In [32]:
# Count matches
match_counts = defaultdict(int)

matched_text = []
for label, pattern in zip(topic_labels, compiled_patterns):
    for text in cleaned_place_text:
        if pattern.search(text):
            matched_text.append(text)
    # count unique strings in matched_text
    # Remove duplicates
    unique_match = set(matched_text)
    # print(f"Unique matches for {label}: {unique_match}")
    match_counts[label] = len(unique_match)
    matched_text = []

# Create dataframe
df = pd.DataFrame({
    "Topic": topic_labels,
    "Match Count": [match_counts[label] for label in topic_labels]
})

In [33]:
df

,Topic,Match Count
0,character & setting,100
1,scale,31
2,density,42
3,heritage,22
4,housing mix,4
5,affordability,56
6,amenity impact,227
7,infrastructure,74
8,traffic / highways,136
9,air pollution,158


In [34]:
# Find examples of comments matching the regex for topic 15
topic_index = 13  # Index for "drainage / flooding"
pattern = compiled_patterns[topic_index]
# Extract comments that match the pattern
matching_comments = [text for text in cleaned_place_text if pattern.search(text)]

# Display a few examples
print(topic_labels[topic_index]+'\n ----')
for comment in matching_comments[:10]:  # Adjust the number of examples as needed
    print(comment)

consultation
 ----
WOODGRANGE SOCIETY AND RESIDENTS ASSOCIATION Chair Michael Striesow ViceChair Erica RadoniDrew 109 Claremont Road 43 Hampton Road Forest Gate Forest Gate London E7 0PY London E7 0PD Tel 020 8279 6818 Tel 020 8221 0700 Nick Fenwick Director of Planning Development London Borough of Newham 1000 Dockside Road London E16 2QU 14th April 2021 Dear Nick Fenwick Planning Application 2002849FUL The WSRA firstly wish to make it clear that residents are not opposed to the redevelopment of Durning Hall especially as it has been left to deteriorate for more than a decade. Apologies for the late response but it has been very difficult due to the restrictions caused by the pandemic to gather and collate views and coordinate a reply. However the loss of community facilities and the proposed 10storey height of the block fronting Woodgrange Road are of great concern to many. The WSRA therefore wish to comment and raise objections to the planning application to redevelop Durning Hall o

### Fuzzy regex match 

Try a similar match but using fuzzy matching - which allows for up to 2 errors in a word to count as a match.

In [ ]:
# Function to build fuzzy regex patterns allowing up to 2 edits
def build_fuzzy_patterns(topic_list, max_errors=2):
    compiled_patterns = []
    for keywords in topic_list:
        # Create a pattern that allows up to `max_errors` edits
        fuzzy_keywords = [fr"({regex.escape(word)}){{e<={max_errors}}}" for word in keywords]
        pattern = regex.compile("|".join(fuzzy_keywords), flags=regex.IGNORECASE)
        compiled_patterns.append(pattern)
    return compiled_patterns

# Compile fuzzy regex patterns
compiled_patterns = build_fuzzy_patterns(topic_list, max_errors=2)

In [ ]:
# Count matches
match_counts = defaultdict(int)

for label, pattern in zip(topic_labels, compiled_patterns):
    matched_text = set()  # Track unique matches per topic
    for text in cleaned_place_text:
        if pattern.search(text):
            matched_text.add(text)
    match_counts[label] = len(matched_text)

# Create dataframe
df_fuzzy = pd.DataFrame({
    "Topic": topic_labels,
    "Match Count": [match_counts[label] for label in topic_labels]
})

In [36]:
df_fuzzy

,Topic,Match Count
0,character & setting,300
1,scale,472
2,density,242
3,heritage,37
4,housing mix,23
5,affordability,60
6,amenity impact,495
7,infrastructure,407
8,traffic / highways,154
9,air pollution,183


In [37]:
# return some examples of comments matching the fuzzy regex for topic 15
topic_index = 13  # Index for "consultation"
pattern = compiled_patterns[topic_index]
# Extract comments that match the pattern
matching_comments = [text for text in cleaned_place_text if pattern.search(text)]
# Display a few examples
print(topic_labels[topic_index]+'\n ----')
for comment in matching_comments[:10]:  # Adjust the number of examples as needed
    print(comment)

consultation
 ----
WOODGRANGE SOCIETY AND RESIDENTS ASSOCIATION Chair Michael Striesow ViceChair Erica RadoniDrew 109 Claremont Road 43 Hampton Road Forest Gate Forest Gate London E7 0PY London E7 0PD Tel 020 8279 6818 Tel 020 8221 0700 Nick Fenwick Director of Planning Development London Borough of Newham 1000 Dockside Road London E16 2QU 14th April 2021 Dear Nick Fenwick Planning Application 2002849FUL The WSRA firstly wish to make it clear that residents are not opposed to the redevelopment of Durning Hall especially as it has been left to deteriorate for more than a decade. Apologies for the late response but it has been very difficult due to the restrictions caused by the pandemic to gather and collate views and coordinate a reply. However the loss of community facilities and the proposed 10storey height of the block fronting Woodgrange Road are of great concern to many. The WSRA therefore wish to comment and raise objections to the planning application to redevelop Durning Hall o

### Cosine similarity of embeddings

Now use the pre-trained Transformer to create embeddings of both the comment_text as well as the lists for each topic. 

Then use cosine similarity to compare them - counting a match if the cosine similarity is above a given threshold. 

In [38]:
# model = SentenceTransformer("all-MiniLM-L6-v2")  # or fine-tune your own on similarity tasks
# AutoTokenizer.from_pretrained("all-MiniLM-L6-v2")

In [ ]:
# Load the model and tokenizer

model_checkpoint = "../outputs/nlp_fine_tuning/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModel.from_pretrained(model_checkpoint)  # NOT AutoModelForMaskedLM

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSdpaAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [40]:
def get_embedding(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    # Mean pooling of last hidden state
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings.squeeze()

In [ ]:
# # Note this uses [CLS] token instead of mean pooling 
# # - which should be better at capturing sentence-level meaning 
# # note: this is definitely worse

# def get_embedding_cls(text, model, tokenizer):
#     inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
#     with torch.no_grad():
#         outputs = model(**inputs)
#     cls_embedding = outputs.last_hidden_state[:, 0, :]  # [CLS] token
#     return cls_embedding.squeeze()

In [ ]:
# Get topic embeddings
topic_embeddings = []
for topic_keywords in topic_list:
    combined_text = " ".join(topic_keywords)
    emb = get_embedding(combined_text, model, tokenizer)
    topic_embeddings.append(emb)

# get comment_text embeddings
text_embeddings = [get_embedding(text, model, tokenizer) for text in cleaned_place_text]

### Calculate the cosine similarity 

Calculate the cosine similarity between the topic lists and the comment text. 

The threshold parameter here is very sensitive. threshold = 0.8 seems about right, lower and it counts everything as a match, higher and it counts nothing as a match. 

In [ ]:
match_counts = {label: 0 for label in topic_labels}
threshold = 0.80  # adjust based on sensitivity

for emb in text_embeddings:
    for label, topic_emb in zip(topic_labels, topic_embeddings):
        similarity = cosine_similarity(emb.unsqueeze(0), topic_emb.unsqueeze(0))
        if similarity.item() > threshold:
            match_counts[label] += 1

In [46]:
df = pd.DataFrame({
    "Topic": topic_labels,
    "Semantic Match Count": [match_counts[label] for label in topic_labels]
})

In [47]:
df

,Topic,Semantic Match Count
0,character & setting,24
1,scale,5
2,density,291
3,heritage,1
4,housing mix,0
5,affordability,73
6,amenity impact,90
7,infrastructure,0
8,traffic / highways,2
9,air pollution,2


Find some matching items 

In [48]:
def find_matching_texts_for_topic(topic_keywords, cleaned_place_text, model, tokenizer, threshold=0.6, top_n=5):
    # Combine keywords into a single string and get topic embedding
    topic_text = " ".join(topic_keywords)
    topic_embedding = get_embedding(topic_text, model, tokenizer)
    
    matches = []
    
    for text in cleaned_place_text:
        text_embedding = get_embedding(text, model, tokenizer)
        similarity = cosine_similarity(topic_embedding.unsqueeze(0), text_embedding.unsqueeze(0)).item()
        if similarity >= threshold:
            matches.append((text, similarity))
    
    # Sort by similarity and return top N
    matches.sort(key=lambda x: x[1], reverse=True)
    return matches[:top_n]


In [49]:
topic_index = 11  
topic_keywords = topic_list[topic_index]

matched_texts = find_matching_texts_for_topic(topic_keywords, cleaned_place_text, model, tokenizer)

for text, score in matched_texts:
    print(f"Similarity: {score:.2f} | Text: {text}")


Similarity: 0.86 | Text: Lack of community space Not meeting affordable housing limit Not protecting small business
Similarity: 0.85 | Text: Im objecting this application because of the following Loss of light or overshadowing school and buildings around Overlookingloss of privacy Adequacy of parkingloadingturning Construction noise dust and disruption Loss of trees and green space overstretching current facilities pool gym and etc. Effect on surrounding building Layout and density of building
Similarity: 0.85 | Text: Too crowded lack of green space Noise pollution while developing Overstretching the services especially for DLR station services inside the area.
Similarity: 0.84 | Text: Credible reasons for objection is increase air pollution light and noise pollution. Increased traffic and congestion.
Similarity: 0.84 | Text: The development is already crowded and I object the application due to the below reasons Landscaping Road access Adequacy of parkingloadingturning Traffic generat

In [50]:
topic_index = 15  
topic_keywords = topic_list[topic_index]

matched_texts = find_matching_texts_for_topic(topic_keywords, cleaned_place_text, model, tokenizer)

for text, score in matched_texts:
    print(f"Similarity: {score:.2f} | Text: {text}")


Similarity: 0.76 | Text: We are oppose to this development due to lack of access and increase traffic flow as it will be the main access for the proposed new buildings. This will create noise pollution congestion and general loss to amenity to residents and cause additional parking issues on the estate roads. Health and safety concerns. Noise of construction and congestion to residents unacceptable to residents. Concerns with utilise ie. sewer and water pipes the system is old and will continue to fail causing further disruption and environmental and health issued.
Similarity: 0.76 | Text: Im objecting this application because of the following Loss of light or overshadowing school and buildings around Overlookingloss of privacy Adequacy of parkingloadingturning Construction noise dust and disruption Loss of trees and green space overstretching current facilities pool gym and etc. Effect on surrounding building Layout and density of building
Similarity: 0.76 | Text: Credible reasons for

## Zero-shot classification 

In [61]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Device set to use mps:0


In [ ]:
# Define the candidate labels
candidate_labels = [
    "character & setting",
    "scale",
    "density",
    "heritage",
    "housing mix",
    "affordability",
    "amenity impact",
    "infrastructure",
    "traffic / highways",
    "air pollution",
    "noise pollution",
    "green space loss",
    "parking loss",
    "highway safety",
    "consultation",
    "accessibility", 
    "drainage / flooding",
    "food"
]
# Get predictions for each comment
predictions = []
for text in cleaned_place_text:
    result = classifier(text, candidate_labels, multi_label=True)
    predictions.append(result)
# Extract the labels and scores
predicted_labels = []
predicted_scores = []
for result in predictions:
    labels = result['labels']
    scores = result['scores']
    predicted_labels.append(labels)
    predicted_scores.append(scores)
# Create a DataFrame to store the results
df_predictions = pd.DataFrame({
    "comment_text": cleaned_place_text,
    "predicted_labels": predicted_labels,
    "predicted_scores": predicted_scores
})



In [72]:
# add column for the top label
df_predictions['top_label'] = df_predictions['predicted_labels'].apply(lambda x: x[0:3])
# add column for the top score
df_predictions['top_score'] = df_predictions['predicted_scores'].apply(lambda x: x[0:3])

In [73]:
df_predictions

,comment_text,predicted_labels,predicted_scores,top_label,top_score
0,The new development will increase noise pollut...,"[scale, housing mix, density, traffic / highwa...","[0.7560450434684753, 0.7202120423316956, 0.484...","[scale, housing mix, density]","[0.7560450434684753, 0.7202120423316956, 0.484..."
1,This will reduce our sunlight significantly wi...,"[green space loss, scale, density, housing mix...","[0.9924995303153992, 0.9790248274803162, 0.862...","[green space loss, scale, density]","[0.9924995303153992, 0.9790248274803162, 0.862..."
2,We were told that there will be no new constru...,"[amenity impact, consultation, infrastructure,...","[0.9827752113342285, 0.9530326128005981, 0.918...","[amenity impact, consultation, infrastructure]","[0.9827752113342285, 0.9530326128005981, 0.918..."
3,I am sorry but I must object to the proposed d...,"[scale, green space loss, amenity impact, dens...","[0.9063330888748169, 0.8874211311340332, 0.884...","[scale, green space loss, amenity impact]","[0.9063330888748169, 0.8874211311340332, 0.884..."
4,This area was planned to be a green space for ...,"[green space loss, amenity impact, density, sc...","[0.9902729988098145, 0.9859445095062256, 0.866...","[green space loss, amenity impact, density]","[0.9902729988098145, 0.9859445095062256, 0.866..."
...,...,...,...,...,...
509,The current proposal reduces facilities to a s...,"[amenity impact, infrastructure, accessibility...","[0.993895411491394, 0.9890903234481812, 0.9152...","[amenity impact, infrastructure, accessibility]","[0.993895411491394, 0.9890903234481812, 0.9152..."
510,Dear Sirs With regard to the above I am specif...,"[green space loss, amenity impact, character &...","[0.9955369234085083, 0.9929925799369812, 0.990...","[green space loss, amenity impact, character &...","[0.9955369234085083, 0.9929925799369812, 0.990..."
511,Whilst an update to this space is welcomed the...,"[amenity impact, character & setting, heritage...","[0.9868689775466919, 0.9046593904495239, 0.800...","[amenity impact, character & setting, heritage]","[0.9868689775466919, 0.9046593904495239, 0.800..."
512,I am the founder and teacher of a martial arts...,"[housing mix, affordability, accessibility, co...","[0.9741450548171997, 0.9730532765388489, 0.949...","[housing mix, affordability, accessibility]","[0.9741450548171997, 0.9730532765388489, 0.949..."


In [74]:
print(df_predictions['comment_text'].iloc[2])
print(df_predictions['predicted_labels'].iloc[2])

We were told that there will be no new construction in Royal Wharf when we were deciding to buy our apartment this was a key factor in the final decision. New construction in a so near distance will be very noisy and dusty. Plus will create more pressure on the already suffering utilitiesamenities.
['amenity impact', 'consultation', 'infrastructure', 'character & setting', 'air pollution', 'scale', 'affordability', 'density', 'housing mix', 'heritage', 'green space loss', 'accessibility', 'drainage / flooding', 'traffic / highways', 'parking', 'highway safety']


In [75]:
# save the predictions to a csv file
df_predictions.to_csv('../outputs/zero_shot_predictions.csv', index=False)